# Qualitative Evaluation: Selected Examples

This notebook presents four representative commands selected from the quanitative evaluation of the grounded language system. The examples should illustrate the qualitative differences between the rule-based parser, the learned intent classifier, the learned slot tagger, and the optional Stage 3 Hugging Face model checker.

The selected commands test:

1. **Paraphrased movement and intent recognition:**


   A multi-step movement request expressed without the parser’s most direct command wording.

2. **Complex reference and target extraction:**


   A command containing several shoe attributes, a relation used to identify the source object, and a separate destination.

3. **Ambiguous reference handling:**


   A deliberately underspecified reference that matches more than one object and should therefore remain unresolved unless additional information is available.

4. **Natural-language command with a direct tool ID:**


   A varied cleaning expression that requires intent recognition, shoe-attribute extraction, dirt-type recognition, and resolution of a specifically named cleaning utensil.


Each command is executed in a newly created world with the same fixed random seed (205). This ensures that all parser configurations receive an identical world state and that earlier commands do not influence later comparisons. The examples are first executed with the three Stage 1 and Stage 2 parser configurations. They are then repeated with the Stage 3 model checker enabled. The outputs demonstrate both improvements and remaining limitations.

In [1]:
import sys
import importlib
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import dialogue, parsers, stage3, world

EVALUATION_SEED = 205

In [2]:
# ----------------------------
# Qualitative evaluation helper
# -----------------------------

# Display the fixed initial world used for every qualitative comparison
demo_world = world.World.create_random(
    seed=EVALUATION_SEED
)
print("#" * 100 + "\nINITIAL WORLD\n" + "#" * 100 + "\n")
print(demo_world.describe())
print("\n" + "#" * 100)



def run_demo(parser, commands, model_checker=None):
    """Run selected commands with one parser configuration.

    Each command receives a newly generated world with the same fixed seed so
    parser configurations can be compared under identical world conditions.
    """
    
    model_phrase = "" if model_checker is None else " with the stage 3 model checking"
    print(f"RUNNING DEMO USING: {parser.__class__.__name__}{model_phrase}\n")

    for number, command in enumerate(commands, start=1):
        # Reset the world so previous actions cannot influence this example
        demo_world = world.World.create_random(seed=EVALUATION_SEED)
        print("#" * 100 + f"\nCOMMAND {number}: {command}\n" + "#" * 100)
        try:
            result = dialogue.interpret_and_act(
                world=demo_world,
                utterance=command,
                parser=parser,
                model_checker=model_checker,
            )
            print(dialogue.format_dialogue_output(result))
        except Exception as error:
            print(f"ERROR: {type(error).__name__}: {error}")

        print()



DEMO_COMMANDS = [
    "Take the wet boot off the top shelf and put it into the drying area.",
    "Move the low wet canvas boot beside the clean blue leather sneaker to the middle shelf.",
    "Pick up the white shoe.",
    "Could you remove the grass from the slightly dirty white rubber trainer with brush_1?",
]

####################################################################################################
INITIAL WORLD
####################################################################################################

Shoes:
- boot_1: wet black canvas boot (h=low, clean=medium_dirty, dirt=mud, material=good, sole=loose, imp=unprotected, loc=top_shelf)
- boot_2: dry white leather boot (h=high, clean=medium_dirty, dirt=mud, material=scratched, sole=loose, imp=unprotected, loc=bottom_shelf)
- sandal_1: dry red rubber sandal (h=mid, clean=medium_dirty, dirt=dust, material=cracked, sole=damaged, imp=protected, loc=floor_box)
- sandal_2: dry blue rubber sandal (h=mid, clean=medium_dirty, dirt=oil, material=scratched, sole=worn, imp=protected, loc=bottom_shelf)
- sandal_3: dry green canvas sandal (h=mid, clean=very_dirty, dirt=mud, material=scratched, sole=intact, imp=partly_protected, loc=floor_box)
- sneaker_1: wet blue leather sneaker (h=low, clean=clean, material=cracked, sole=worn, imp=pr

In [3]:
run_demo(
    parser=parsers.RuleBasedParser(), 
    commands=DEMO_COMMANDS
)

RUNNING DEMO USING: RuleBasedParser

####################################################################################################
COMMAND 1: Take the wet boot off the top shelf and put it into the drying area.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: PICK_UP

I could not find the requested shoe.
I looked for this shoe:
- object class: shoe
- shoe type: boot
- dry status: wet
- relations:
  - relation: on top_shelf
  - relation: inside drying_area

####################################################################################################
COMMAND 2: Move the low wet canvas boot beside the clean blue leather sneaker to the middle shelf.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: MOV

In [4]:
run_demo(
    parser=parsers.IntentClassifierParser(), 
    commands=DEMO_COMMANDS
)

RUNNING DEMO USING: IntentClassifierParser

####################################################################################################
COMMAND 1: Take the wet boot off the top shelf and put it into the drying area.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: MOVE

Resolved 'shoe' to boot_1.

____________________ACTION RESULT____________________

Action sequence:
1. Picked up boot_1 from top_shelf.
2. Put boot_1 down at drying_area. The shoe is now drying.

####################################################################################################
COMMAND 2: Move the low wet canvas boot beside the clean blue leather sneaker to the middle shelf.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command

In [5]:
run_demo(
    parser=parsers.SlotTaggerParser(), 
    commands=DEMO_COMMANDS
)

RUNNING DEMO USING: SlotTaggerParser

####################################################################################################
COMMAND 1: Take the wet boot off the top shelf and put it into the drying area.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: MOVE

Resolved 'shoe' to boot_1.

____________________ACTION RESULT____________________

Action sequence:
1. Picked up boot_1 from top_shelf.
2. Put boot_1 down at drying_area. The shoe is now drying.

####################################################################################################
COMMAND 2: Move the low wet canvas boot beside the clean blue leather sneaker to the middle shelf.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: M

In [6]:
import os
from getpass import getpass

hf_token = os.getenv("HF_TOKEN") or getpass(
    "Enter your Hugging Face token: "
)

Enter your Hugging Face token:  ········


In [7]:
run_demo(
    parser=parsers.RuleBasedParser(), 
    commands=DEMO_COMMANDS,
    model_checker=stage3.ModelChecker(
        hf_token=hf_token,
        verbose=True, 
         confidence_threshold=0.90
    )
)

RUNNING DEMO USING: RuleBasedParser with the stage 3 model checking

####################################################################################################
COMMAND 1: Take the wet boot off the top shelf and put it into the drying area.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: PICK_UP
Stage 3 intent check: corrected intent 'PICK_UP' -> 'MOVE' 
Reason: The command specifies moving a wet boot from the top shelf to the drying area, which requires a destination and object transfer. (Confidence: 0.95)

Resolved 'shoe' to boot_1.

____________________ACTION RESULT____________________

Action sequence:
1. Picked up boot_1 from top_shelf.
2. Put boot_1 down at drying_area. The shoe is now drying.

####################################################################################################
COMMAND 2: Move the low wet canvas boot

In [8]:
run_demo(
    parser=parsers.IntentClassifierParser(), 
    commands=DEMO_COMMANDS,
    model_checker=stage3.ModelChecker(
        hf_token=hf_token,
        verbose=True, 
         confidence_threshold=0.90
    )
)

RUNNING DEMO USING: IntentClassifierParser with the stage 3 model checking

####################################################################################################
COMMAND 1: Take the wet boot off the top shelf and put it into the drying area.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: MOVE
Stage 3 intent check: kept intent 'MOVE' 
Reason: The command specifies moving a wet boot from the top shelf to the drying area, which matches the MOVE intent with a clear destination. (Confidence: 0.95)

Resolved 'shoe' to boot_1.

____________________ACTION RESULT____________________

Action sequence:
1. Picked up boot_1 from top_shelf.
2. Put boot_1 down at drying_area. The shoe is now drying.

####################################################################################################
COMMAND 2: Move the low wet canvas boot beside 

In [9]:
run_demo(
    parser=parsers.SlotTaggerParser(), 
    commands=DEMO_COMMANDS,
    model_checker=stage3.ModelChecker(
        hf_token=hf_token,
        verbose=True, 
         confidence_threshold=0.90
    )
)

RUNNING DEMO USING: SlotTaggerParser with the stage 3 model checking

####################################################################################################
COMMAND 1: Take the wet boot off the top shelf and put it into the drying area.
####################################################################################################
____________________SYSTEM MESSAGES____________________

I interpreted your command as: MOVE
Stage 3 intent check: kept intent 'MOVE' 
Reason: The command specifies moving a wet boot from the top shelf to the drying area, which matches the MOVE intent with a clear destination. (Confidence: 0.95)

Resolved 'shoe' to boot_1.

____________________ACTION RESULT____________________

Action sequence:
1. Picked up boot_1 from top_shelf.
2. Put boot_1 down at drying_area. The shoe is now drying.

####################################################################################################
COMMAND 2: Move the low wet canvas boot beside the cl